# 뉴럴넷으로 패션 아이템 구분하기
Fashion MNIST 데이터셋과 앞서 배운 인공신경망을 이용하여 패션아이템을 구분해봅니다.

In [1]:
import torch # torch 임포트
import torch.nn as nn # nn 임포트
import torch.optim as optim # optim 임포트
import torch.nn.functional as F # functional 임포트
from torchvision import transforms, datasets # torchvision 데이터셋 임포트
import os # os 임포트
import matplotlib.pyplot as plt # 시각화용 임포트
import numpy as np # numpy 임포트
from sklearn.metrics import confusion_matrix # 혼동 행렬 임포트
import seaborn as sns # seaborn 임포트

In [2]:
# GPU 사용 가능 여부 확인
USE_CUDA = torch.cuda.is_available() # CUDA 사용 가능 여부
DEVICE = torch.device("cuda" if USE_CUDA else "cpu") # 장치 설정

In [3]:
# 학습 파라미터 설정
EPOCHS = 30        # 총 학습 횟수
BATCH_SIZE = 64    # 배치 크기
DEVICE # 현재 장치 출력

In [4]:
# 모델 저장 경로 설정
MODEL_PATH = './model/fashion_mnist_model.pth' # 모델 파일 경로

## 데이터셋 불러오기

In [5]:
transform = transforms.Compose([ # 데이터 전처리 정의
    transforms.ToTensor() # 텐서 변환
])

In [6]:
trainset = datasets.FashionMNIST( # 학습 데이터셋 로드
    root      = './.data/', 
    train     = True,
    download  = True,
    transform = transform
)
testset = datasets.FashionMNIST( # 테스트 데이터셋 로드
    root      = './.data/', 
    train     = False,
    download  = True,
    transform = transform
)

In [7]:
train_loader = torch.utils.data.DataLoader( # 학습 로더 생성
    dataset     = trainset,
    batch_size  = BATCH_SIZE,
    shuffle     = True,
)
test_loader = torch.utils.data.DataLoader( # 테스트 로더 생성
    dataset     = testset,
    batch_size  = BATCH_SIZE,
    shuffle     = True,
)

## 뉴럴넷으로 Fashion MNIST 학습하기
입력 `x` 는 `[배치크기, 색, 높이, 넓이]`로 이루어져 있습니다.
`x.size()`를 해보면 `[64, 1, 28, 28]`이라고 표시되는 것을 보실 수 있습니다.
Fashion MNIST에서 이미지의 크기는 28 x 28, 색은 흑백으로 1 가지 입니다.
그러므로 입력 x의 총 특성값 갯수는 28 x 28 x 1, 즉 784개 입니다.
우리가 사용할 모델은 3개의 레이어를 가진 인공신경망 입니다. 

In [8]:
class Net(nn.Module): # 신경망 클래스 정의
    def __init__(self):
        super(Net, self).__init__()
        # 3개의 완전연결층 정의
        self.fc1 = nn.Linear(784, 256)  # 입력 -> 은닉1
        self.fc2 = nn.Linear(256, 128)  # 은닉1 -> 은닉2
        self.fc3 = nn.Linear(128, 10)   # 은닉2 -> 출력

    def forward(self, x): # 순전파 함수
        x = x.view(-1, 784)            # 데이터 평탄화
        x = F.relu(self.fc1(x))        # fc1 + ReLU
        x = F.relu(self.fc2(x))        # fc2 + ReLU
        x = self.fc3(x)                # fc3 출력
        return x

## 모델 준비하기
`to()` 함수는 모델의 파라미터들을 지정한 곳으로 보내는 역할을 합니다.
일반적으로 CPU 1개만 사용할 경우 필요는 없지만,
GPU를 사용하고자 하는 경우 `to("cuda")`로 지정하여 GPU로 보내야 합니다.
지정하지 않을 경우 계속 CPU에 남아 있게 되며 빠른 훈련의 이점을 누리실 수 없습니다.
최적화 알고리즘으로 파이토치에 내장되어 있는 `optim.SGD`를 사용하겠습니다.

In [9]:
model        = Net().to(DEVICE) # 모델 생성 및 장치 이동
optimizer    = optim.SGD(model.parameters(), lr=0.01) # SGD 최적화기 설정

## 학습하기

In [10]:
def train(model, train_loader, optimizer): # 학습 함수 정의
    model.train()  # 모델을 학습 모드로 설정
    for batch_idx, (data, target) in enumerate(train_loader): # 미니배치 반복
        data, target = data.to(DEVICE), target.to(DEVICE)  # 장치 이동
        optimizer.zero_grad()  # 기울기 초기화
        output = model(data)   # 출력 계산
        loss = F.cross_entropy(output, target)  # 손실 계산
        loss.backward()        # 역전파
        optimizer.step()       # 가중치 업데이트

## 테스트하기

In [11]:
def evaluate(model, test_loader): # 평가 함수 정의
    model.eval() # 평가 모드 설정
    test_loss = 0 # 손실 합계
    correct = 0 # 맞춘 개수
    all_preds = [] # 예측 리스트
    all_targets = [] # 실제 리스트
    with torch.no_grad(): # 기울기 계산 비활성화
        for data, target in test_loader:
            data, target = data.to(DEVICE), target.to(DEVICE)
            output = model(data)
            
            test_loss += F.cross_entropy(output, target,
                                       reduction='sum').item() # 손실 합산
            
            pred = output.max(1, keepdim=True)[1] # 예측 클래스
            correct += pred.eq(target.view_as(pred)).sum().item() # 정답 확인
            
            # 예측값과 실제값 저장
            all_preds.extend(pred.cpu().numpy())
            all_targets.extend(target.cpu().numpy())

    test_loss /= len(test_loader.dataset) # 평균 손실
    test_accuracy = 100. * correct / len(test_loader.dataset) # 정확도
    return test_loss, test_accuracy, all_preds, all_targets

## 코드 돌려보기
자, 이제 모든 준비가 끝났습니다. 코드를 돌려서 실제로 훈련이 되는지 확인해봅시다!

In [12]:
# 모델 저장 디렉토리 생성
os.makedirs('./model', exist_ok=True) # 폴더 생성

In [13]:
# 학습 기록을 저장할 리스트
train_losses = [] # 학습 손실
train_accuracies = [] # 학습 정확도
test_losses = [] # 테스트 손실
test_accuracies = [] # 테스트 정확도

In [14]:
# 클래스 이름 정의
CLASSES = { # 클래스 맵핑
    0: 'T-shirt/top',
    1: 'Trouser',
    2: 'Pullover',
    3: 'Dress',
    4: 'Coat',
    5: 'Sandal',
    6: 'Shirt',
    7: 'Sneaker',
    8: 'Bag',
    9: 'Ankle boot'
}

In [15]:
# 학습 및 평가
for epoch in range(1, EPOCHS + 1): # 에포크 반복
    train(model, train_loader, optimizer) # 학습 수행
    test_loss, test_accuracy, all_preds, all_targets = evaluate(model, test_loader) # 평가 수행
    
    # 학습 기록 저장
    test_losses.append(test_loss) # 기록 저장
    test_accuracies.append(test_accuracy) # 기록 저장
    
    print('[{}] Test Loss: {:.4f}, Accuracy: {:.2f}%'.format(
          epoch, test_loss, test_accuracy)) # 로그 출력
    
    # 매 에포크마다 모델 저장
    torch.save({ # 체크포인트 저장
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': test_loss,
        'accuracy': test_accuracy
    }, MODEL_PATH)

In [16]:
print('모델이 저장되었습니다:', MODEL_PATH) # 저장 완료 메시지

In [17]:
# 학습 커브 그리기
plt.figure(figsize=(12, 4)) # 그래프 영역 설정

In [18]:
# 손실 그래프
plt.subplot(1, 2, 1) # 서브플롯 설정
plt.plot(test_losses, 'b-', label='Test Loss') # 손실 플롯
plt.title('Model Loss') # 제목 설정
plt.xlabel('Epoch') # X축 라벨
plt.ylabel('Loss') # Y축 라벨
plt.legend() # 범례 표시

In [19]:
# 정확도 그래프
plt.subplot(1, 2, 2) # 서브플롯 설정
plt.plot(test_accuracies, 'r-', label='Test Accuracy') # 정확도 플롯
plt.title('Model Accuracy') # 제목 설정
plt.xlabel('Epoch') # X축 라벨
plt.ylabel('Accuracy (%)') # Y축 라벨
plt.legend() # 범례 표시

In [20]:
plt.tight_layout() # 레이아웃 조정
plt.savefig('./model/learning_curves.png') # 그래프 저장
plt.show() # 그래프 출력

In [21]:
# 혼동 행렬 계산 및 시각화
cm = confusion_matrix(all_targets, all_preds) # 혼동 행렬 계산
plt.figure(figsize=(10, 8)) # 그래프 영역 설정
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=list(CLASSES.values()),
            yticklabels=list(CLASSES.values())) # 히트맵 시각화
plt.title('Confusion Matrix') # 제목 설정
plt.xlabel('Predicted') # X축 라벨
plt.ylabel('True') # Y축 라벨
plt.xticks(rotation=45, ha='right') # X축 라벨 회전
plt.yticks(rotation=0) # Y축 라벨 설정
plt.tight_layout() # 레이아웃 조정
plt.savefig('./model/confusion_matrix.png') # 이미지 저장
plt.show() # 그래프 출력

In [22]:
# 저장된 모델 불러오기
def load_model(model_path): # 모델 로드 함수
    if os.path.exists(model_path): # 파일 존재 확인
        checkpoint = torch.load(model_path) # 체크포인트 로드
        model.load_state_dict(checkpoint['model_state_dict']) # 모델 가중치 복원
        optimizer.load_state_dict(checkpoint['optimizer_state_dict']) # 최적화기 상태 복원
        epoch = checkpoint['epoch'] # 에포크 정보
        loss = checkpoint['loss'] # 손실 정보
        accuracy = checkpoint['accuracy'] # 정확도 정보
        print(f'모델을 불러왔습니다. (에포크: {epoch}, 손실: {loss:.4f}, 정확도: {accuracy:.2f}%)')
        return model, optimizer
    else:
        print('저장된 모델이 없습니다.')
        return None, None

In [23]:
# 저장된 모델 불러오기 예시
loaded_model, loaded_optimizer = load_model(MODEL_PATH) # 로드 실행

In [24]:
# 불러온 모델로 테스트
if loaded_model is not None: # 로드 성공 시
    test_loss, test_accuracy, all_preds, all_targets = evaluate(loaded_model, test_loader) # 평가 재실행
    print(f'불러온 모델 테스트 결과 - 손실: {test_loss:.4f}, 정확도: {test_accuracy:.2f}%') # 결과 출력